<a href="https://colab.research.google.com/github/arauch6363-crypto/pt/blob/main/PT_github_2026_today.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install selenium
!pip install unidecode
!pip install fastparquet
!pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 8.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.6 MB/s eta 0:00:00


In [ ]:
#set mode

mode = 'get_todays_races'
#mode = 'get_past_results'
#mode = 'get_both'

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Cell 3 - Imports
from datetime import date, datetime, timedelta
import pytz
import json
import re
import pandas as pd
from unidecode import unidecode
import numpy as np
import sys
import fastparquet

ModuleNotFoundError: No module named 'unidecode'

In [ ]:

# Cell 4 - Helper functions
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
nest_asyncio.apply()

async def get_web_content_async(url, retries=3):
    for attempt in range(retries):
        try:
            async with async_playwright() as p:
                browser = await p.chromium.launch(headless=True)
                page = await browser.new_page()
                try:
                    await page.goto(url, wait_until='commit', timeout=60000)
                    await page.wait_for_selector('#__NEXT_DATA__', state='attached', timeout=60000)
                    content = await page.evaluate('document.getElementById("__NEXT_DATA__").textContent')
                    return json.loads(content)
                finally:
                    await browser.close()
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for {url}: {e}")
            if attempt == retries - 1:
                raise
            await asyncio.sleep(3)

def get_web_content(url, driver=None):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(get_web_content_async(url))

def web_driver():
    return None

def get_operator_data_odds(data, operators_priority=['PMU', 'PMU.fr', 'genybet']):
    for operator in operators_priority:
        for race in data:
            if race.get('operator') == operator:
                return race.get('runners', {})
    return None

def get_operator_data_dividends(data, operators_priority=['PMU', 'PMU.fr', 'genybet']):
    for operator in operators_priority:
        for race in data:
            if race.get('operator') == operator:
                return race.get('betDividends', {})
    return None

def generate_top_5_table(df, group_column):
    df_wins = df[df['ranking'] == 1]
    total_runs = df.groupby(group_column).size()
    win_count = df_wins.groupby(group_column).size()
    result = pd.DataFrame({
        'wins': win_count,
        'runs': total_runs
    }).fillna(0)
    result['win_percentage'] = (result['wins'] / result['runs']) * 100
    top_5 = result.sort_values(by='wins', ascending=False).head(20)
    top_5['formatted'] = top_5.apply(
        lambda row: f"{row.name} {int(row['runs'])}/{int(row['wins'])} {row['win_percentage']:.0f}%", axis=1
    )
    return top_5['formatted']

In [ ]:
def get_today():

    driver = None  # not needed with Playwright

    try:
        loadDate = datetime.today().strftime('%Y-%m-%d')
        url = f'https://www.paris-turf.com/programme-courses/{loadDate}'
        my_tz = pytz.timezone('Europe/Berlin')
        timestamp_start = datetime.now(my_tz).strftime('%Y-%m-%d %H:%M:%S')

        print('loadDate:', loadDate)
        print("Running today's data collection...")

        props = get_web_content(url)
        races = props['props']['pageProps']['initialState']['raceCardsState']['races'][loadDate]
        meetings = props['props']['pageProps']['initialState']['raceCardsState']['meetings'][loadDate]

        df_races_tdy = pd.DataFrame(races)
        df_meetings_tdy = pd.DataFrame(meetings)

        merged_df = pd.merge(
            df_meetings_tdy[['id', 'name', 'country']],
            df_races_tdy,
            left_on='id',
            right_on='meetingId',
            suffixes=('_meeting', '_race')
        )

        filtered_df = merged_df[
            (merged_df['country'] == 'FR') &
            (merged_df['discipline'] != 'Trot') &
            (merged_df['specialty'] == 'P') &
            (merged_df['breed'].isnull())
        ]

        required_columns = ['date', 'name_meeting', 'meetingId', 'trackCode', 'name_race', 'id_race', 'specialty', 'class', 'type', 'sex',
                            'surface', 'going', 'penetrometer', 'number', 'distance', 'isPremium', 'breed', 'totalPrize',
                            'uuid', 'direction', 'rail', 'winningPost', 'minAge', 'maxAge', 'discipline', 'winnerTimeKm', 'winnerTime']

        final_df = filtered_df.reindex(columns=required_columns, fill_value=np.nan)

        def clean_name(name):
            name = unidecode(name.lower())
            name = name.replace(' ', '-').replace('(', '').replace(')', '')
            return name

        final_df['cleaned_name_race'] = final_df['name_race'].apply(clean_name)
        final_df['cleaned_name_meeting'] = final_df['name_meeting'].apply(clean_name)
        final_df['race_url'] = 'https://www.paris-turf.com/course/' + final_df['cleaned_name_meeting'] + '-' + final_df['cleaned_name_race'] + '-idc-' + final_df['uuid']
        final_df.drop(columns=['cleaned_name_race', 'cleaned_name_meeting'], inplace=True)

        race_df_tdy = final_df.reset_index(drop=True)
        race_df_tdy['runners_loaded'] = False
        race_df_tdy.to_parquet("./races_tdy.parquet", engine='pyarrow', index=None)

        runners_dfs = []
        webTips_dfs = []
        odds_dfs = []

        for i in range(len(race_df_tdy)):
            props = get_web_content(race_df_tdy['race_url'][i])

            runners = props['props']['pageProps']['initialState']['raceCardsState']['runners'][str(race_df_tdy['id_race'][i])]
            df_runners = pd.DataFrame(runners)

            try:
                webTips = props['props']['pageProps']['initialState'].get('currentPageState', {}).get('webTips', [])
                df_webTips = pd.DataFrame(webTips).reset_index()
            except:
                df_webTips = pd.DataFrame()

            try:
                odds = get_operator_data_odds(
                    props['props']['pageProps']['initialState'].get('currentPageState', {}).get('betinRaceOdd', {}).get('odds', {})
                )
                df_odds = pd.DataFrame(odds).reset_index()
            except:
                df_odds = pd.DataFrame()

            keys = ['horseId', 'isRunnerState', 'meetingId', 'age', 'hood', 'breederName', 'shoeingFront', 'isEngaged', 'jockeyName', 'draw', 'saddle', 'isSupplemented', 'isRunning', 'weightKg', 'raceDirection', 'numberOfPlaces', 'ownerName', 'raceId', 'uuid', 'raceSpeciality', 'noShoesFirstTime',
                    'raceTotalPrize', 'ranking', 'jockeyUUID', 'totalPrize', 'horseName', 'horseSir', 'trainerUUID', 'meetingName', 'horseUUID', 'ownerUUID', 'trainerName', 'protectionFirstTime', 'raceName', 'jockeyAllowance', 'tongueTie', 'raceIsTQQ', 'sex', 'shoeingBack',
                    'totalWinningPrize','breederId', 'margin', 'jockeyChanged', 'coloursPng', 'shoeing', 'bestImpression', 'comment', 'isPremium', 'blinkers', 'jockeyId', 'raceType', 'ownerId', 'handicapRatingKg', 'horseDam', 'weightChanged', 'claimRating',
                    'trainerId', 'blinkersFirstTime','raceNumber']

            keys2 = ['meetingId', 'raceId', 'text', 'tips']
            keys_odds = ['horseId', 'horseNumber', 'liveOdd', 'referenceOdd', 'isFavorite', 'runnerId', 'liveOddDateTime', 'referenceOddDateTime', 'runnerStatus', 'runnerSlug', 'horseName']

            if not df_runners.empty:
                df_runners = df_runners.reindex(columns=keys, fill_value=np.nan)
                runners_dfs.append(df_runners)

            if not df_webTips.empty:
                df_webTips = df_webTips.reindex(columns=keys2, fill_value=np.nan)
                webTips_dfs.append(df_webTips)

            if not df_odds.empty:
                df_odds = df_odds.reindex(columns=keys_odds, fill_value=np.nan)
                df_odds['meetingId'] = race_df_tdy['meetingId'][i]
                df_odds['raceId'] = race_df_tdy['id_race'][i]
                odds_dfs.append(df_odds)

            print(race_df_tdy['race_url'][i], df_runners.shape)

            if df_runners.shape[0] < 2:
                sys.exit("Notebook Execution Stopped: Less than 2 runners")

        if runners_dfs:
            df_runners_tdy = pd.concat(runners_dfs, ignore_index=True)
            df_runners_tdy.to_parquet("./runners_tdy.parquet", engine='fastparquet', index=None)

        if webTips_dfs:
            df_webTips_tdy = pd.concat(webTips_dfs, ignore_index=True)
            df_webTips_tdy.to_parquet("./webTips_tdy.parquet", engine='pyarrow', index=None)

        if odds_dfs:
            df_odds_tdy = pd.concat(odds_dfs, ignore_index=True)
            df_odds_tdy.to_parquet("./odds_tdy.parquet", engine='pyarrow', index=None)

        return True
    finally:
        pass  # driver.quit() not needed with Playwright

In [ ]:
def get_today():

    # Initialize the driver once for this get_today execution
    driver = web_driver()

    try:
        loadDate = datetime.today().strftime('%Y-%m-%d')
        url = f'https://www.paris-turf.com/programme-courses/{loadDate}'
        my_tz = pytz.timezone('Europe/Berlin')
        timestamp_start = datetime.now(my_tz).strftime('%Y-%m-%d %H:%M:%S')

        print('loadDate:', loadDate)
        print("Running today's data collection...")

        props = get_web_content(url, driver)
        races = props['props']['pageProps']['initialState']['raceCardsState']['races'][loadDate]
        meetings = props['props']['pageProps']['initialState']['raceCardsState']['meetings'][loadDate]

        df_races_tdy = pd.DataFrame(races)
        df_meetings_tdy = pd.DataFrame(meetings)

        merged_df = pd.merge(
            df_meetings_tdy[['id', 'name', 'country']],
            df_races_tdy,
            left_on='id',
            right_on='meetingId',
            suffixes=('_meeting', '_race')
        )

        filtered_df = merged_df[
            (merged_df['country'] == 'FR') &
            (merged_df['discipline'] != 'Trot') &
            (merged_df['specialty'] == 'P') &
            (merged_df['breed'].isnull())
        ]

        required_columns = ['date', 'name_meeting', 'meetingId', 'trackCode', 'name_race', 'id_race', 'specialty', 'class', 'type', 'sex',
                            'surface', 'going', 'penetrometer', 'number', 'distance', 'isPremium', 'breed', 'totalPrize',
                            'uuid', 'direction', 'rail', 'winningPost', 'minAge', 'maxAge', 'discipline', 'winnerTimeKm', 'winnerTime']

        final_df = filtered_df.reindex(columns=required_columns, fill_value=np.nan)

        def clean_name(name):
            name = unidecode(name.lower())
            name = name.replace(' ', '-').replace('(', '').replace(')', '')
            return name

        final_df['cleaned_name_race'] = final_df['name_race'].apply(clean_name)
        final_df['cleaned_name_meeting'] = final_df['name_meeting'].apply(clean_name)
        final_df['race_url'] = 'https://www.paris-turf.com/course/' + final_df['cleaned_name_meeting'] + '-' + final_df['cleaned_name_race'] + '-idc-' + final_df['uuid']
        final_df.drop(columns=['cleaned_name_race', 'cleaned_name_meeting'], inplace=True)

        race_df_tdy = final_df.reset_index(drop=True)
        race_df_tdy['runners_loaded'] = False
        race_df_tdy.to_parquet("/content/drive/MyDrive/PT/races_tdy.parquet", engine='pyarrow', index=None)

        runners_dfs = []
        webTips_dfs = []
        odds_dfs = []


        for i in range(len(race_df_tdy)):
            props = get_web_content(race_df_tdy['race_url'][i], driver)


            # Runners
            runners = props['props']['pageProps']['initialState']['raceCardsState']['runners'][str(race_df_tdy['id_race'][i])]
            df_runners = pd.DataFrame(runners)

            # WebTips (optional)
            try:
                webTips = props['props']['pageProps']['initialState'].get('currentPageState', {}).get('webTips', [])
                df_webTips = pd.DataFrame(webTips).reset_index()
            except:
                df_webTips = pd.DataFrame()

            # Odds (optional)
            try:
                odds = get_operator_data_odds(
                    props['props']['pageProps']['initialState'].get('currentPageState', {}).get('betinRaceOdd', {}).get('odds', {})
                )
                df_odds = pd.DataFrame(odds).reset_index()
            except:
                df_odds = pd.DataFrame()

            # Define the keys for runners and webTips
            keys = ['horseId', 'isRunnerState', 'meetingId', 'age', 'hood', 'breederName', 'shoeingFront', 'isEngaged', 'jockeyName', 'draw', 'saddle', 'isSupplemented', 'isRunning', 'weightKg', 'raceDirection', 'numberOfPlaces', 'ownerName', 'raceId', 'uuid', 'raceSpeciality', 'noShoesFirstTime',
                    'raceTotalPrize', 'ranking', 'jockeyUUID', 'totalPrize', 'horseName', 'horseSir', 'trainerUUID', 'meetingName', 'horseUUID', 'ownerUUID', 'trainerName', 'protectionFirstTime', 'raceName', 'jockeyAllowance', 'tongueTie', 'raceIsTQQ', 'sex', 'shoeingBack',
                    'totalWinningPrize','breederId', 'margin', 'jockeyChanged', 'coloursPng', 'shoeing', 'bestImpression', 'comment', 'isPremium', 'blinkers', 'jockeyId', 'raceType', 'ownerId', 'handicapRatingKg', 'horseDam', 'weightChanged', 'claimRating',
                    'trainerId', 'blinkersFirstTime','raceNumber']

            keys2 = ['meetingId', 'raceId', 'text', 'tips']

            keys_odds = ['horseId', 'horseNumber', 'liveOdd', 'referenceOdd', 'isFavorite', 'runnerId', 'liveOddDateTime', 'referenceOddDateTime', 'runnerStatus', 'runnerSlug', 'horseName']

            if not df_runners.empty:
                df_runners = df_runners.reindex(columns=keys, fill_value=np.nan)
                runners_dfs.append(df_runners)

            if not df_webTips.empty:
                df_webTips = df_webTips.reindex(columns=keys2, fill_value=np.nan)
                webTips_dfs.append(df_webTips)

            if not df_odds.empty:
                df_odds = df_odds.reindex(columns=keys_odds, fill_value=np.nan)
                df_odds['meetingId'] = race_df_tdy['meetingId'][i]
                df_odds['raceId'] = race_df_tdy['id_race'][i]
                odds_dfs.append(df_odds)

            print(race_df_tdy['race_url'][i], df_runners.shape)

            if df_runners.shape[0] < 2:
                sys.exit("Notebook Execution Stopped: Less than 2 runners")

        # Save runners
        if runners_dfs:
            df_runners_tdy = pd.concat(runners_dfs, ignore_index=True)
            df_runners_tdy.to_parquet("/content/drive/MyDrive/PT/runners_tdy.parquet", engine='fastparquet', index=None)

        # Save webTips
        if webTips_dfs:
            df_webTips_tdy = pd.concat(webTips_dfs, ignore_index=True)
            df_webTips_tdy.to_parquet("/content/drive/MyDrive/PT/webTips_tdy.parquet", engine='pyarrow', index=None)

        # Save odds
        if odds_dfs:
            df_odds_tdy = pd.concat(odds_dfs, ignore_index=True)
            df_odds_tdy.to_parquet("/content/drive/MyDrive/PT/odds_tdy.parquet", engine='pyarrow', index=None)

        return True
    finally:
        driver.quit() # Ensure the driver is quit even if errors occur

In [ ]:
if mode in ('get_todays_races', 'get_both'):
  get_today()